---
## FASE 3: Preprocesamiento de Datos

### 9. Remoción de Density (data leakage) y separación X / y

In [ ]:
# Density se excluye por data leakage:
# fue usada para calcular BodyFat mediante la fórmula de Siri,
# por lo que incluirla equivale a filtrar la respuesta en los features.

FEATURES = [c for c in df_clean.columns if c not in ['BodyFat', 'Density']]
TARGET   = 'BodyFat'

X = df_clean[FEATURES].copy()
y = df_clean[TARGET].copy()

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Target  : {TARGET}')
print(f'Shape X : {X.shape}   |   Shape y : {y.shape}')

### 10. Data Splitting — Train / Test (80/20)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(f'Train : {X_train.shape[0]} observaciones  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Test  : {X_test.shape[0]}  observaciones  ({X_test.shape[0]/len(X)*100:.1f}%)')

### 11. Escalado de features — StandardScaler

> **Regla de oro:** `fit` solo en train, `transform` en train y test.  
> Hacer `fit` sobre todo el dataset introduce *data leakage* (la media y desvío del test contaminarían el escalado).

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_sc = scaler.fit_transform(X_train)   # fit + transform sobre train
X_test_sc  = scaler.transform(X_test)        # solo transform sobre test

# Verificación: media ≈ 0 y std ≈ 1 en train
print('Verificación sobre X_train escalado:')
print(f'  Media  → min: {X_train_sc.mean(axis=0).min():.4f}  |  max: {X_train_sc.mean(axis=0).max():.4f}')
print(f'  Desvío → min: {X_train_sc.std(axis=0).min():.4f}  |  max: {X_train_sc.std(axis=0).max():.4f}')
print('✅ Escalado correcto')

---
## FASE 4: Modelado — Algoritmos Lineales

### 12. Métricas de evaluación

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def evaluar_modelo(nombre, y_true, y_pred):
    """
    Calcula RMSE, MAE, MAPE y R² y devuelve un dict con los resultados.
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100   # en %
    r2   = r2_score(y_true, y_pred)

    print(f'\n{'='*45}')
    print(f'  {nombre}')
    print(f'{'='*45}')
    print(f'  RMSE : {rmse:.4f}  (misma unidad que BodyFat, %)')
    print(f'  MAE  : {mae:.4f}')
    print(f'  MAPE : {mape:.2f}%')
    print(f'  R²   : {r2:.4f}')

    return {'Modelo': nombre, 'RMSE': round(rmse,4),
            'MAE': round(mae,4), 'MAPE (%)': round(mape,2), 'R²': round(r2,4)}

resultados = []   # acumulador para tabla comparativa final
print('✅ Función de evaluación definida')

### 13. Modelo Baseline — DummyRegressor (predice siempre la media)

Todo modelo de ML debe superar este umbral mínimo para aportar valor.

In [ ]:
from sklearn.dummy import DummyRegressor

dummy = DummyRegressor(strategy='mean')
dummy.fit(X_train_sc, y_train)
y_pred_dummy = dummy.predict(X_test_sc)

print(f'Media de BodyFat en train: {y_train.mean():.2f}%')
print(f'(El baseline predice ese valor para todas las observaciones de test)')

res_dummy = evaluar_modelo('Baseline (media)', y_test, y_pred_dummy)
resultados.append(res_dummy)

### 14. Regresión Lineal (OLS)

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

res_lr = evaluar_modelo('Regresión Lineal (OLS)', y_test, y_pred_lr)
resultados.append(res_lr)

In [ ]:
# Coeficientes — Regresión Lineal
coef_lr = pd.DataFrame({
    'Feature'    : FEATURES,
    'Coeficiente': lr.coef_
}).sort_values('Coeficiente', key=abs, ascending=False)

print(f'Intercepto: {lr.intercept_:.4f}')
print('\nCoeficientes (ordenados por magnitud absoluta):')
print(coef_lr.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if v > 0 else 'salmon' for v in coef_lr['Coeficiente']]
ax.barh(coef_lr['Feature'], coef_lr['Coeficiente'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coeficientes — Regresión Lineal (OLS)', fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente (datos escalados)')
plt.tight_layout()
plt.show()

### 15. Modelo Regularizado — Lasso (LassoCV)

Lasso usa penalización L1, que fuerza a cero los coeficientes de variables poco relevantes.  
`LassoCV` selecciona el mejor `alpha` por cross-validation interno, evitando la búsqueda manual.

In [ ]:
from sklearn.linear_model import LassoCV

lasso_cv = LassoCV(
    alphas=np.logspace(-4, 2, 100),   # grilla de alphas a evaluar
    cv=5,
    random_state=42,
    max_iter=10_000
)
lasso_cv.fit(X_train_sc, y_train)
y_pred_lasso = lasso_cv.predict(X_test_sc)

print(f'Alpha óptimo (LassoCV): {lasso_cv.alpha_:.6f}')
res_lasso = evaluar_modelo('Lasso (LassoCV)', y_test, y_pred_lasso)
resultados.append(res_lasso)

In [ ]:
# Coeficientes Lasso — visualizar cuáles fueron llevados a cero
coef_lasso = pd.DataFrame({
    'Feature'    : FEATURES,
    'Coeficiente': lasso_cv.coef_
}).sort_values('Coeficiente', key=abs, ascending=False)

n_cero    = (coef_lasso['Coeficiente'] == 0).sum()
n_activos = (coef_lasso['Coeficiente'] != 0).sum()
print(f'Variables activas (coef ≠ 0): {n_activos}')
print(f'Variables eliminadas (coef = 0): {n_cero}')
print()
print(coef_lasso.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['steelblue' if v > 0 else ('salmon' if v < 0 else 'lightgrey')
          for v in coef_lasso['Coeficiente']]
ax.barh(coef_lasso['Feature'], coef_lasso['Coeficiente'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Coeficientes — Lasso  (α = {lasso_cv.alpha_:.4f})', fontsize=13, fontweight='bold')
ax.set_xlabel('Coeficiente (datos escalados)')
plt.tight_layout()
plt.show()

### 16. Comparación de modelos lineales — Tabla resumen

In [ ]:
df_resultados = pd.DataFrame(resultados)

# Highlight: mejor R² y menor RMSE
print('=== Tabla comparativa — Fase 4 ===')
print(df_resultados.to_string(index=False))

# Gráfico de barras comparativo
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, metrica, mejor in zip(axes, ['RMSE', 'MAE', 'R²'], ['min', 'min', 'max']):
    vals   = df_resultados[metrica]
    labels = df_resultados['Modelo']
    idx_best = vals.idxmin() if mejor == 'min' else vals.idxmax()
    colors = ['gold' if i == idx_best else 'steelblue' for i in range(len(vals))]
    ax.bar(labels, vals, color=colors, edgecolor='white')
    ax.set_title(metrica, fontsize=13, fontweight='bold')
    ax.set_xticklabels(labels, rotation=20, ha='right', fontsize=9)
    for i, v in enumerate(vals):
        ax.text(i, v + (max(vals)-min(vals))*0.02, f'{v:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.suptitle('Comparación de modelos — Fase 4', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 17. Gráfico de residuos — Predicciones vs. Valores reales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

modelos_preds = [
    ('Baseline (media)'       , y_pred_dummy),
    ('Regresión Lineal (OLS)' , y_pred_lr),
    ('Lasso'                  , y_pred_lasso),
]

for ax, (nombre, y_pred) in zip(axes, modelos_preds):
    residuos = y_test - y_pred
    ax.scatter(y_pred, residuos, alpha=0.5, s=25, color='steelblue')
    ax.axhline(0, color='red', linewidth=1.2, linestyle='--')
    ax.set_title(nombre, fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicción (% BodyFat)')
    ax.set_ylabel('Residuo')

plt.suptitle('Gráfico de Residuos por Modelo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()